# Diabetes Prediction - Deep Learning From Scratch

Notebook này chạy với dataset `diabetes.csv` giống trong slide.

Dataset có 8 input features và 1 target column, nên model dùng đúng kiến trúc `8 -> 16 -> 8 -> 1`.

Model vẫn theo tinh thần slide:

- No TensorFlow
- No PyTorch
- No Keras
- No Scikit-learn
- ReLU, sigmoid, binary cross-entropy, backpropagation, gradient descent bằng NumPy

## 1. Load Data

Load `diabetes.csv` bằng `np.loadtxt`, rồi tách `X = data[:, :8]` và `y = data[:, 8]` giống slide.

In [29]:
# ============================================================
# DEEP LEARNING FROM SCRATCH
# Diabetes Prediction
#
# NO TensorFlow
# NO PyTorch
# NO Keras
# NO Scikit-learn
#
# Only NumPy is used for the neural network.
# ============================================================

from pathlib import Path
import numpy as np

# ============================================================
# 1. LOAD DATA
# ============================================================

DATA_CANDIDATES = [
    Path("../data/diabetes.csv"),
    Path("data/diabetes.csv"),
    Path("src/Assigment 03/diabetes/data/diabetes.csv"),
]

data_path = next((path for path in DATA_CANDIDATES if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find diabetes.csv")

data = np.loadtxt(
    data_path,
    delimiter=",",
    skiprows=1
)

X = data[:, :8]
y = data[:, 8].reshape(-1, 1)

print("Data path:", data_path.resolve())
print("X shape:", X.shape)
print("y shape:", y.shape)

Data path: /Users/macos/Docs/Kì 1 năm 4/PTHTTM/src/Assigment 03/diabetes/data/diabetes.csv
X shape: (768, 8)
y shape: (768, 1)


## 2. Train/Test Split

Giữ cách split 80/20 bằng permutation như slide.

In [30]:
# ============================================================
# 2. TRAIN / TEST SPLIT
# ============================================================

np.random.seed(42)

indices = np.random.permutation(len(X))
split = int(0.8 * len(X))

train_idx = indices[:split]
test_idx = indices[split:]

X_train = X[train_idx]
y_train = y[train_idx]

X_test = X[test_idx]
y_test = y[test_idx]

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

Training samples: 614
Testing samples : 154


## 3. Normalization

Chuẩn hóa bằng mean/std của training set, giống slide.

In [31]:
# ============================================================
# 3. NORMALIZATION
# ============================================================

mean = X_train.mean(axis=0)
std = X_train.std(axis=0) + 1e-8

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

## 4. Activation Functions

In [32]:
# ============================================================
# 4. ACTIVATION FUNCTIONS
# ============================================================

def relu(x):
    return np.maximum(0, x)


def relu_derivative(x):
    return (x > 0).astype(float)


def sigmoid(x):
    x = np.clip(x, -50, 50)
    return 1.0 / (1.0 + np.exp(-x))

## 5. Initialize Network

Dataset `diabetes.csv` có 8 input features, nên kiến trúc giữ đúng như slide: `8 -> 16 -> 8 -> 1`.

In [33]:
# ============================================================
# 5. INITIALIZE NETWORK
# ============================================================

np.random.seed(42)

# Layer 1: 8 -> 16
W1 = (
    np.random.randn(8, 16)
    * np.sqrt(2.0 / 8)
)
b1 = np.zeros((1, 16))

# Layer 2: 16 -> 8
W2 = (
    np.random.randn(16, 8)
    * np.sqrt(2.0 / 16)
)
b2 = np.zeros((1, 8))

# Layer 3: 8 -> 1
W3 = (
    np.random.randn(8, 1)
    * np.sqrt(2.0 / 8)
)
b3 = np.zeros((1, 1))

print("W1 shape:", W1.shape)
print("W2 shape:", W2.shape)
print("W3 shape:", W3.shape)

W1 shape: (8, 16)
W2 shape: (16, 8)
W3 shape: (8, 1)


## 6. Forward Propagation

In [34]:
# ============================================================
# 6. FORWARD PROPAGATION
# ============================================================

def forward(X):
    # Layer 1
    z1 = X @ W1 + b1
    h1 = relu(z1)

    # Layer 2
    z2 = h1 @ W2 + b2
    h2 = relu(z2)

    # Layer 3
    z3 = h2 @ W3 + b3
    y_hat = sigmoid(z3)

    cache = {
        "X": X,
        "z1": z1,
        "h1": h1,
        "z2": z2,
        "h2": h2,
        "z3": z3,
        "y_hat": y_hat,
    }

    return y_hat, cache

## 7. Loss

In [35]:
# ============================================================
# 7. LOSS
# ============================================================

def binary_cross_entropy(y, y_hat):
    eps = 1e-8
    y_hat = np.clip(
        y_hat,
        eps,
        1 - eps
    )

    loss = -np.mean(
        y * np.log(y_hat)
        +
        (1 - y)
        * np.log(1 - y_hat)
    )

    return loss

## 8. Backpropagation

In [36]:
# ============================================================
# 8. BACKPROPAGATION
# ============================================================

def backward(y, cache):
    X = cache["X"]
    z1 = cache["z1"]
    h1 = cache["h1"]
    z2 = cache["z2"]
    h2 = cache["h2"]
    yhat = cache["y_hat"]

    n = len(X)

    # --------------------------------------------------------
    # Layer 3
    # --------------------------------------------------------
    dz3 = (yhat - y) / n
    dW3 = h2.T @ dz3
    db3 = np.sum(
        dz3,
        axis=0,
        keepdims=True
    )

    # --------------------------------------------------------
    # Layer 2
    # --------------------------------------------------------
    dh2 = dz3 @ W3.T
    dz2 = (
        dh2
        * relu_derivative(z2)
    )
    dW2 = h1.T @ dz2
    db2 = np.sum(
        dz2,
        axis=0,
        keepdims=True
    )

    # --------------------------------------------------------
    # Layer 1
    # --------------------------------------------------------
    dh1 = dz2 @ W2.T
    dz1 = (
        dh1
        * relu_derivative(z1)
    )
    dW1 = X.T @ dz1
    db1 = np.sum(
        dz1,
        axis=0,
        keepdims=True
    )

    gradients = {
        "dW1": dW1,
        "db1": db1,
        "dW2": dW2,
        "db2": db2,
        "dW3": dW3,
        "db3": db3,
    }

    return gradients

## 9. Training

Giữ full-batch gradient descent, `learning_rate = 0.01`, `epochs = 1000` như slide.

In [37]:
# ============================================================
# 9. TRAINING
# ============================================================

learning_rate = 0.01
epochs = 1000

for epoch in range(epochs):
    # Forward
    y_hat, cache = forward(
        X_train
    )

    # Loss
    loss = binary_cross_entropy(
        y_train,
        y_hat
    )

    # Backward
    gradients = backward(
        y_train,
        cache
    )

    # Update
    W1 -= (
        learning_rate
        * gradients["dW1"]
    )
    b1 -= (
        learning_rate
        * gradients["db1"]
    )

    W2 -= (
        learning_rate
        * gradients["dW2"]
    )
    b2 -= (
        learning_rate
        * gradients["db2"]
    )

    W3 -= (
        learning_rate
        * gradients["dW3"]
    )
    b3 -= (
        learning_rate
        * gradients["db3"]
    )

    if epoch % 100 == 0:
        print(
            f"Epoch {epoch:4d} | "
            f"Loss = {loss:.4f}"
        )

Epoch    0 | Loss = 0.8080
Epoch  100 | Loss = 0.6232
Epoch  200 | Loss = 0.5539
Epoch  300 | Loss = 0.5130
Epoch  400 | Loss = 0.4873
Epoch  500 | Loss = 0.4704
Epoch  600 | Loss = 0.4587
Epoch  700 | Loss = 0.4498
Epoch  800 | Loss = 0.4424
Epoch  900 | Loss = 0.4360


## 10. Test

In [38]:
# ============================================================
# 10. TEST
# ============================================================

y_prob, _ = forward(
    X_test
)

y_pred = (
    y_prob >= 0.5
).astype(int)

## 11. Accuracy

In [39]:
# ============================================================
# 11. ACCURACY
# ============================================================

accuracy = np.mean(
    y_pred == y_test
)

print()
print("==============================")
print("TEST RESULTS")
print("==============================")
print("Accuracy:", accuracy)


TEST RESULTS
Accuracy: 0.7662337662337663


## 12. Confusion Matrix

In [40]:
# ============================================================
# 12. CONFUSION MATRIX
# ============================================================

TP = np.sum(
    (y_pred == 1)
    &
    (y_test == 1)
)
TN = np.sum(
    (y_pred == 0)
    &
    (y_test == 0)
)
FP = np.sum(
    (y_pred == 1)
    &
    (y_test == 0)
)
FN = np.sum(
    (y_pred == 0)
    &
    (y_test == 1)
)

print()
print("Confusion Matrix")
print("-----------------")
print("TP:", TP)
print("TN:", TN)
print("FP:", FP)
print("FN:", FN)


Confusion Matrix
-----------------
TP: 33
TN: 85
FP: 11
FN: 25


## 13. Precision / Recall / F1

In [41]:
# ============================================================
# 13. PRECISION / RECALL / F1
# ============================================================

precision = (
    TP
    /
    (TP + FP + 1e-8)
)
recall = (
    TP
    /
    (TP + FN + 1e-8)
)
f1 = (
    2 * precision
    * recall
    /
    (precision + recall + 1e-8)
)

print()
print("Precision:", precision)
print("Recall   :", recall)
print("F1       :", f1)


Precision: 0.7499999998295455
Recall   : 0.5689655171432818
F1       : 0.6470588184967322


## 14. Example Predictions

In [42]:
# ============================================================
# 14. EXAMPLE PREDICTIONS
# ============================================================

print()
print("Example predictions")
print("--------------------")

for i in range(
    min(10, len(X_test))
):
    print(
        f"Patient {i + 1:2d} | "
        f"Probability = "
        f"{y_prob[i, 0]:.3f} | "
        f"Prediction = "
        f"{y_pred[i, 0]} | "
        f"Actual = "
        f"{int(y_test[i, 0])}"
    )


Example predictions
--------------------
Patient  1 | Probability = 0.053 | Prediction = 0 | Actual = 0
Patient  2 | Probability = 0.096 | Prediction = 0 | Actual = 0
Patient  3 | Probability = 0.575 | Prediction = 1 | Actual = 0
Patient  4 | Probability = 0.486 | Prediction = 0 | Actual = 0
Patient  5 | Probability = 0.748 | Prediction = 1 | Actual = 1
Patient  6 | Probability = 0.594 | Prediction = 1 | Actual = 0
Patient  7 | Probability = 0.149 | Prediction = 0 | Actual = 0
Patient  8 | Probability = 0.220 | Prediction = 0 | Actual = 0
Patient  9 | Probability = 0.415 | Prediction = 0 | Actual = 1
Patient 10 | Probability = 0.349 | Prediction = 0 | Actual = 0
